# Classificação Supervisionada com K-Means no Dataset Dry Bean

Este notebook apresenta a implementação de um classificador supervisionado baseado no algoritmo K-Means, aplicado ao dataset Dry Bean. Todas as etapas são explicadas e o código é comentado para facilitar o entendimento.

## 1. Importação das Bibliotecas

Importamos as bibliotecas necessárias para manipulação de dados, visualização e implementação do K-Means.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

## 2. Carregamento e Pré-processamento dos Dados

Carregamos o dataset Dry Bean, removemos valores ausentes, codificamos variáveis categóricas e normalizamos os dados.

In [ ]:
# Substitua o caminho pelo local correto do seu arquivo Dry Bean
bean_df = pd.read_excel('data/Dry_Bean_Dataset.xlsx')

# Remover valores ausentes
bean_df = bean_df.dropna()

# Codificar variáveis categóricas
for col in bean_df.select_dtypes(include='object').columns:
    bean_df[col] = LabelEncoder().fit_transform(bean_df[col].astype(str))

# Separar atributos e rótulo
X = bean_df.drop('Class', axis=1).values
y = bean_df['Class'].values

# Normalizar os dados
scaler = StandardScaler()
X = scaler.fit_transform(X)

## 3. Divisão dos Dados em Treino e Teste

Dividimos o conjunto de dados em 70% para treinamento e 30% para teste, garantindo a estratificação das classes.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

## 4. Definição do Número de Clusters (Método do Cotovelo)

Utilizamos o método do cotovelo para determinar o número ideal de clusters para o K-Means.

In [ ]:
distortions = []
K = range(2, 11)
for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X_train)
    distortions.append(kmeans.inertia_)

plt.figure(figsize=(8,4))
plt.plot(K, distortions, 'bx-')
plt.xlabel('Número de clusters')
plt.ylabel('Distortion (Inertia)')
plt.title('Método do Cotovelo para o K-Means')
plt.savefig('img/elbow_kmeans_drybean.png')
plt.show()

## 5. Implementação do Classificador K-Means Supervisionado

O classificador associa cada cluster ao rótulo mais frequente no conjunto de treino. O código é comentado para facilitar o entendimento.

In [ ]:
class KMeansSupervisionado:
    def __init__(self, n_clusters=2, random_state=0):
        self.n_clusters = n_clusters
        self.random_state = random_state
        self.kmeans = KMeans(n_clusters=n_clusters, random_state=random_state)
        self.cluster_labels_ = None

    def fit(self, X, y):
        # Ajusta o KMeans e associa cada cluster ao rótulo mais frequente
        clusters = self.kmeans.fit_predict(X)
        self.cluster_labels_ = []
        for i in range(self.n_clusters):
            mask = (clusters == i)
            if np.any(mask):
                label = np.bincount(y[mask]).argmax()
            else:
                label = -1
            self.cluster_labels_.append(label)

    def predict(self, X):
        # Prediz os rótulos para os dados de entrada com base nos clusters
        clusters = self.kmeans.predict(X)
        return np.array([self.cluster_labels_[c] for c in clusters])

    def evaluate(self, X, y_true):
        # Avalia o classificador retornando acurácia e matriz de confusão
        y_pred = self.predict(X)
        acc = accuracy_score(y_true, y_pred)
        cm = confusion_matrix(y_true, y_pred)
        return acc, cm

## 6. Treinamento e Avaliação do Classificador

Treinamos o classificador no conjunto de treino e avaliamos no conjunto de teste, mostrando acurácia e matriz de confusão.

In [ ]:
# Defina o número de clusters de acordo com o método do cotovelo
n_clusters = 7  # Exemplo, ajuste conforme o gráfico do cotovelo

clf = KMeansSupervisionado(n_clusters=n_clusters, random_state=42)
clf.fit(X_train, y_train)

acc, cm = clf.evaluate(X_test, y_test)
print(f'Acurácia: {acc:.4f}')
print('Matriz de Confusão:')
print(cm)

plt.figure(figsize=(6,5))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Matriz de Confusão - KMeans (Dry Bean)')
plt.colorbar()
plt.ylabel('Verdadeiro')
plt.xlabel('Predito')
plt.savefig('img/confusion_matrix_kmeans_drybean.png')
plt.show()

## 7. Repetição dos Experimentos

Repita o experimento 30 vezes, variando a semente, e salve as acurácias para análise estatística.

In [ ]:
acuracias = []
for seed in range(1, 31):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=seed, stratify=y)
    clf = KMeansSupervisionado(n_clusters=n_clusters, random_state=seed)
    clf.fit(X_train, y_train)
    acc, _ = clf.evaluate(X_test, y_test)
    acuracias.append(acc)

acuracias = np.array(acuracias)
print(f'Acurácia média: {acuracias.mean():.4f}')
print(f'Desvio padrão: {acuracias.std():.4f}')

plt.figure(figsize=(8,4))
plt.plot(range(1,31), acuracias, marker='o')
plt.xlabel('Repetição')
plt.ylabel('Acurácia')
plt.title('Acurácia por repetição - KMeans (Dry Bean)')
plt.savefig('img/accuracy_repetitions_kmeans_drybean.png')
plt.show()

## 8. Análise dos Resultados

Comente os resultados obtidos, destacando a acurácia média, o desvio padrão e possíveis dificuldades do classificador K-Means no dataset Dry Bean.